In [ ]:
"""
=============================================================================
[Final Robust Version: Modern Transformer with SWA & Full Metrics]

1. Architecture Upgrade (Step 3):
   - Activation: GELU (Gaussian Error Linear Unit) - Standard for LLMs.
   - Positional Encoding: Learnable (Adaptive to specific data patterns).
   - Normalization: Pre-Norm (Better gradient flow).

2. Training Strategy (Step 5):
   - SWA (Stochastic Weight Averaging): Averages weights from the last 
     epoch to flatten the loss landscape and improve generalization.

3. Evaluation (Step 4):
   - Full implementation of BLEU, ROUGE-L, METEOR, and BERTScore.

4. Structure:
   - Strictly 6 steps, fully expanded code for readability.
=============================================================================
"""

import math
import os
import random
import re
import sys
import copy
import time
import unicodedata
import warnings
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel, SWALR

# --- Metrics ---
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

try:
    from bert_score import score as bert_score_func

    HAS_BERTSCORE = False
except ImportError:
    HAS_BERTSCORE = False
    print("[Warning] 'bert-score' not installed. Skipping BERTScore.")

import optuna

warnings.filterwarnings("ignore")


# =============================================================================
# 0. Hardware & System Setup
# =============================================================================
def setup_system():
    if not torch.cuda.is_available():
        sys.exit("CRITICAL: GPU is required.")

    # Enable TF32 for Ampere+ GPUs (Faster FP32 math)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

    # Check NLTK data
    try:
        nltk.data.find("corpora/wordnet")
        nltk.data.find("corpora/omw-1.4")
    except LookupError:
        print("[System] Downloading NLTK data...")
        nltk.download("wordnet", quiet=True)
        nltk.download("omw-1.4", quiet=True)

    print(f"[System] GPU: {torch.cuda.get_device_name(0)}")
    print("[System] Mode: SWA + GELU + Learnable PE")


setup_system()


# =============================================================================
# 1. Configuration
# =============================================================================
class Config:
    SEED = 2025
    DEVICE = torch.device("cuda")

    # Data
    LANG_SRC = "cn"
    LANG_TGT = "eng"
    MAX_LENGTH = 100
    SOS_TOKEN = 0
    EOS_TOKEN = 1
    PAD_TOKEN = 2

    # Training & SWA
    BATCH_SIZE = 128
    NUM_WORKERS = 8

    # Optuna settings
    N_TRIALS = 100
    N_EPOCHS_OPT = 20

    # Full Training settings (Deep Train + SWA)
    N_EPOCHS_FULL = 80
    # Start averaging weights after 75% of training
    SWA_START_EPOCH = 60
    SWA_LR = 0.05  # Learning rate for SWA phase

    CLIP_GRAD = 1.0
    LABEL_SMOOTHING = 0.1


def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(Config.SEED)


# =============================================================================
# 2. Data Processing
# =============================================================================
class Vocab:
    def __init__(self, name):
        self.name = name
        self.word2index = {"SOS": 0, "EOS": 1, "PAD": 2}
        self.index2word = {0: "SOS", 1: "EOS", 2: "PAD"}
        self.n_words = 3

    def add_sentence(self, sentence):
        # CN: Char-level, EN: Word-level
        tokens = sentence if self.name == "cn" else sentence.split()
        for word in tokens:
            if word not in self.word2index:
                self.word2index[word] = self.n_words
                self.index2word[self.n_words] = word
                self.n_words += 1


class DataEngine:
    @staticmethod
    def normalize_string(s):
        s = "".join(
            c
            for c in unicodedata.normalize("NFD", s)
            if unicodedata.category(c) != "Mn"
        )
        s = s.lower().strip()
        s = re.sub(r"([.!?])", r" \1", s)
        s = re.sub(r"[^a-zA-Z\u4e00-\u9fa5.!?]+", r" ", s)
        return s.strip()

    @staticmethod
    def prepare_data():
        filename = f"{Config.LANG_SRC}-{Config.LANG_TGT}.txt"
        if not os.path.exists(filename):
            print("[Warning] Dataset missing. Using dummy data.")
            raw = ["你好\thello", "谢谢\tthank you"] * 200
        else:
            raw = open(filename, encoding="utf-8").read().strip().split("\n")

        pairs = [
            [DataEngine.normalize_string(s) for s in line.split("\t")]
            for line in raw
            if "\t" in line
        ]
        pairs = [
            p
            for p in pairs
            if len(p) == 2
            and len(p[0]) < Config.MAX_LENGTH
            and len(p[1].split()) < Config.MAX_LENGTH
        ]

        inp = Vocab(Config.LANG_SRC)
        out = Vocab(Config.LANG_TGT)

        for p in pairs:
            inp.add_sentence(p[0])
            out.add_sentence(p[1])
        print(f"[Data] Loaded {len(pairs)} pairs. Vocab: {inp.n_words}/{out.n_words}")
        return inp, out, pairs

    @staticmethod
    def collate_fn(batch):
        src_batch, tgt_batch = zip(*batch)
        src_pad = pad_sequence(
            src_batch, padding_value=Config.PAD_TOKEN, batch_first=False
        )
        tgt_pad = pad_sequence(
            tgt_batch, padding_value=Config.PAD_TOKEN, batch_first=False
        )
        return src_pad, tgt_pad


class TransDataset(Dataset):
    def __init__(self, pairs, inp, out, augment=False):
        self.pairs = pairs
        self.inp = inp
        self.out = out
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_txt, tgt_txt = self.pairs[idx]

        s_tokens = src_txt if self.inp.name == "cn" else src_txt.split()
        s_idx = [self.inp.word2index[w] for w in s_tokens]

        # Robustness Augmentation
        if self.augment and len(s_idx) > 4:
            if random.random() < 0.15:  # Swap
                i, j = random.sample(range(len(s_idx)), 2)
                s_idx[i], s_idx[j] = s_idx[j], s_idx[i]
            if random.random() < 0.1:  # Drop
                s_idx.pop(random.randint(0, len(s_idx) - 1))

        t_tokens = tgt_txt if self.out.name == "cn" else tgt_txt.split()
        t_idx = [self.out.word2index[w] for w in t_tokens]

        return torch.tensor(s_idx + [Config.EOS_TOKEN], dtype=torch.long), torch.tensor(
            [Config.SOS_TOKEN] + t_idx + [Config.EOS_TOKEN], dtype=torch.long
        )


# =============================================================================
# 3. Model: Modern Transformer (GELU + Learnable PE)
# =============================================================================
class LearnablePositionalEncoding(nn.Module):
    """
    Optimized: Replaced fixed Sin/Cos with Learnable Embeddings.
    This allows the model to learn position dependencies specific to the dataset.
    """

    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        # We create a lookup table for positions 0...max_len
        self.pe = nn.Embedding(max_len, d_model)

    def forward(self, x):
        # x shape: [Seq_Len, Batch_Size, Embedding_Dim]
        seq_len = x.size(0)
        # Create position indices: [0, 1, 2, ..., seq_len-1]
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(1)
        # Add position embeddings to word embeddings
        x = x + self.pe(positions)
        return self.dropout(x)


class ModernTransformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model,
        nhead,
        num_layers,
        dim_ff,
        dropout,
    ):
        super().__init__()
        self.d_model = d_model

        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)

        # Use Learnable PE
        self.pos_enc = LearnablePositionalEncoding(
            d_model, dropout, Config.MAX_LENGTH + 50
        )

        # Optimized Transformer Block:
        # 1. activation="gelu": Smoother than ReLU, standard in BERT/LLaMA.
        # 2. norm_first=True: Pre-LN is more stable for training deep networks.
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_ff,
            dropout=dropout,
            activation="gelu",
            norm_first=True,
            batch_first=False,
        )

        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

        # Weight Tying: Output projection shares weights with input embedding
        self.tgt_embedding.weight = self.fc_out.weight

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, tgt):
        # Generate Masks
        tgt_seq_len = tgt.shape[0]
        tgt_mask = torch.triu(
            torch.ones(tgt_seq_len, tgt_seq_len, device=Config.DEVICE) * float("-inf"),
            diagonal=1,
        )
        src_pad_mask = (src == Config.PAD_TOKEN).transpose(0, 1)
        tgt_pad_mask = (tgt == Config.PAD_TOKEN).transpose(0, 1)

        # Apply Embeddings + Positional Encoding
        src_emb = self.pos_enc(self.src_embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_enc(self.tgt_embedding(tgt) * math.sqrt(self.d_model))

        # Transformer Pass
        out = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_pad_mask,
            tgt_key_padding_mask=tgt_pad_mask,
            memory_key_padding_mask=src_pad_mask,
        )

        return self.fc_out(out)


# =============================================================================
# 4. Evaluator (Quad-Metric System)
# =============================================================================
class Evaluator:
    @staticmethod
    def beam_search(model, src, out_lang, beam_width=6, max_len=100):
        if isinstance(model, AveragedModel):
            model = model.module
        model.eval()
        with torch.no_grad():
            src_pad_mask = (src == Config.PAD_TOKEN).transpose(0, 1)

            # Encoder Pass
            src_emb = model.pos_enc(model.src_embedding(src) * math.sqrt(model.d_model))
            memory = model.transformer.encoder(
                src_emb, src_key_padding_mask=src_pad_mask
            )

            # Beam Initialization
            beam = [(0.0, [Config.SOS_TOKEN])]

            for _ in range(max_len):
                candidates = []
                for score, seq in beam:
                    if seq[-1] == Config.EOS_TOKEN:
                        candidates.append((score, seq))
                        continue

                    tgt_in = torch.tensor([seq], device=Config.DEVICE).transpose(0, 1)
                    tgt_emb = model.pos_enc(
                        model.tgt_embedding(tgt_in) * math.sqrt(model.d_model)
                    )

                    # Decoder Step
                    out = model.transformer.decoder(
                        tgt_emb, memory, memory_key_padding_mask=src_pad_mask
                    )
                    logits = model.fc_out(out[-1, :])
                    log_probs = F.log_softmax(logits, dim=-1)

                    topk = log_probs.topk(beam_width)
                    for i in range(beam_width):
                        val = topk.values[0][i].item()
                        idx = topk.indices[0][i].item()
                        candidates.append((score + val, seq + [idx]))

                # Sort by score normalized by length
                beam = sorted(
                    candidates, key=lambda x: x[0] / (len(x[1]) ** 0.7), reverse=True
                )[:beam_width]

                if all(s[-1] == Config.EOS_TOKEN for _, s in beam):
                    break

            best_seq = beam[0][1]
            return [
                out_lang.index2word[i]
                for i in best_seq
                if i not in (Config.SOS_TOKEN, Config.EOS_TOKEN, Config.PAD_TOKEN)
            ]

    @staticmethod
    def _calculate_rouge_l_f1(hyp, ref):
        """Calculates ROUGE-L F1 score using LCS."""
        m, n = len(hyp), len(ref)
        if m == 0 or n == 0:
            return 0.0
        dp = [[0] * (n + 1) for _ in range(m + 1)]

        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if hyp[i - 1] == ref[j - 1]:
                    dp[i][j] = dp[i - 1][j - 1] + 1
                else:
                    dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

        lcs = dp[m][n]
        precision = lcs / m if m > 0 else 0
        recall = lcs / n if n > 0 else 0

        if (precision + recall) == 0:
            return 0.0
        return 2 * precision * recall / (precision + recall)

    @staticmethod
    def calculate_all_metrics(
        model, pairs, inp_lang, out_lang, sample_size=150, use_bertscore=False
    ):
        model.eval()
        subset = random.sample(pairs, min(len(pairs), sample_size))

        refs_tokens, hyps_tokens = [], []
        refs_str, hyps_str = [], []

        # 1. Decoding
        for src_txt, tgt_txt in subset:
            s_idx = [
                inp_lang.word2index.get(w, Config.PAD_TOKEN)
                for w in (src_txt if inp_lang.name == "cn" else src_txt.split())
            ]
            src_tensor = torch.tensor(
                s_idx + [Config.EOS_TOKEN], device=Config.DEVICE
            ).unsqueeze(1)

            # Use Greedy search (width=1) for faster validation stats
            pred_toks = Evaluator.beam_search(model, src_tensor, out_lang, beam_width=6)
            ref_toks = tgt_txt.split()

            refs_tokens.append([ref_toks])
            hyps_tokens.append(pred_toks)

            refs_str.append(tgt_txt)
            hyps_str.append(" ".join(pred_toks))

        metrics = {}

        # 2. BLEU
        metrics["bleu"] = corpus_bleu(
            refs_tokens, hyps_tokens, smoothing_function=SmoothingFunction().method1
        )

        # 3. ROUGE-L
        rouge_scores = [
            Evaluator._calculate_rouge_l_f1(h, r[0])
            for h, r in zip(hyps_tokens, refs_tokens)
        ]
        metrics["rouge"] = sum(rouge_scores) / len(rouge_scores) if rouge_scores else 0

        # 4. METEOR
        meteor_scores = [meteor_score(r, h) for r, h in zip(refs_tokens, hyps_tokens)]
        metrics["meteor"] = (
            sum(meteor_scores) / len(meteor_scores) if meteor_scores else 0
        )

        # 5. BERTScore
        if HAS_BERTSCORE and use_bertscore:
            try:
                # Returns P, R, F1. We take F1.
                _, _, F1 = bert_score_func(
                    hyps_str,
                    refs_str,
                    lang="en",
                    model_type="distilbert-base-uncased",
                    device=Config.DEVICE,
                    verbose=False,
                    batch_size=32,
                )
                metrics["bertscore"] = F1.mean().item()
            except Exception as e:
                print(f"[Error] BERTScore calculation failed: {e}")
                metrics["bertscore"] = 0.0
        else:
            metrics["bertscore"] = 0.0

        return metrics


# =============================================================================
# 5. Trainer (SWA Integration)
# =============================================================================
class Trainer:
    def __init__(
        self, model, lr, weight_decay, steps_per_epoch, epochs, pct_start=0.3
    ):  # <--- Update: Added pct_start
        self.model = model
        self.optimizer = optim.AdamW(
            model.parameters(), lr=lr, weight_decay=weight_decay
        )
        self.scaler = torch.amp.GradScaler("cuda")
        self.criterion = nn.CrossEntropyLoss(
            ignore_index=Config.PAD_TOKEN, label_smoothing=Config.LABEL_SMOOTHING
        )

        self.epochs = epochs

        # --- SWA Initialization ---
        self.use_swa = epochs >= Config.SWA_START_EPOCH

        if self.use_swa:
            print(
                f"[Trainer] SWA Enabled. Averaging starts at epoch {Config.SWA_START_EPOCH}."
            )
            self.swa_model = AveragedModel(model)
            self.swa_scheduler = SWALR(self.optimizer, swa_lr=lr * Config.SWA_LR)
        else:
            self.swa_model = None
            self.swa_scheduler = None

        # Standard scheduler with Dynamic Warmup (pct_start)
        self.scheduler = optim.lr_scheduler.OneCycleLR(
            self.optimizer,
            max_lr=lr,
            epochs=epochs,
            steps_per_epoch=steps_per_epoch,
            pct_start=pct_start,  # <--- Update: Use dynamic parameter
        )

    def train_epoch(self, loader, current_epoch):
        self.model.train()
        total_loss = 0

        for src, tgt in loader:
            src, tgt = src.to(Config.DEVICE), tgt.to(Config.DEVICE)
            tgt_in = tgt[:-1, :]
            tgt_out = tgt[1:, :]

            self.optimizer.zero_grad()

            with torch.amp.autocast("cuda"):
                # Forward
                output = self.model(src, tgt_in)
                # Reshape for loss: [Seq*Batch, Vocab]
                loss = self.criterion(
                    output.reshape(-1, output.shape[-1]), tgt_out.reshape(-1)
                )

            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), Config.CLIP_GRAD)
            self.scaler.step(self.optimizer)
            self.scaler.update()

            # --- SWA vs Normal Scheduling Logic ---
            if self.use_swa and current_epoch > Config.SWA_START_EPOCH:
                # Inside SWA phase: we don't step OneCycleLR anymore
                pass
            else:
                # Normal phase
                self.scheduler.step()

            total_loss += loss.item()

        # --- SWA Updates (Per Epoch) ---
        if self.use_swa and current_epoch > Config.SWA_START_EPOCH:
            self.swa_model.update_parameters(self.model)
            self.swa_scheduler.step()

        return total_loss / len(loader)

    def fit(self, loader, val_pairs, inp_lang, out_lang, epochs, trial=None):
        history = {"loss": [], "bleu": [], "rouge": [], "meteor": [], "bertscore": []}

        for epoch in range(1, epochs + 1):
            loss = self.train_epoch(loader, epoch)

            eval_model = self.model
            do_full_metrics = (epoch == epochs) or (trial is None and epoch % 10 == 0)

            metrics = Evaluator.calculate_all_metrics(
                eval_model,
                val_pairs,
                inp_lang,
                out_lang,
                sample_size=150,
                use_bertscore=do_full_metrics,
            )

            history["loss"].append(loss)
            for k in ["bleu", "rouge", "meteor", "bertscore"]:
                history[k].append(metrics[k])

            score = metrics["bleu"] + metrics["meteor"]
            if trial:
                trial.report(score, epoch)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()

            if not trial and (epoch % 10 == 0 or epoch == epochs):
                print(
                    f"Epoch {epoch:03d} | Loss: {loss:.4f} | "
                    f"BLEU: {metrics['bleu']:.3f} | ROUGE: {metrics['rouge']:.3f} | "
                    f"METEOR: {metrics['meteor']:.3f} | BERTScore: {metrics['bertscore']:.3f}"
                )

        if self.use_swa:
            print(">>> Finalizing SWA Model (Updating BatchNorm statistics)...")
            self.final_model = self.swa_model
        else:
            self.final_model = self.model

        return history


# =============================================================================
# 6. Main Flow
# =============================================================================
def objective(trial, inp, out, train_pairs, val_pairs):
    """Optuna Optimization Objective"""
    # --- Architecture Search ---
    nhead = trial.suggest_categorical("nhead", [2, 4, 8])
    base_dim = trial.suggest_categorical("base_dim", [32, 64])
    d_model = base_dim * nhead

    num_layers = trial.suggest_int("num_layers", 1, 4)
    ff_mult = trial.suggest_int("ff_mult", 2, 4)
    dim_ff = d_model * ff_mult

    # --- Optimization Hyperparameters ---
    dropout = trial.suggest_float("dropout", 0.1, 0.4)  # Cap at 0.4 for stability

    # Lowered max LR to 5e-4 to prevent divergence in transformers
    lr = trial.suggest_float("lr", 1e-4, 5e-4, log=True)
    wd = trial.suggest_float("wd", 1e-6, 1e-3, log=True)

    # Search for optimal warmup percentage
    pct_start = trial.suggest_float("pct_start", 0.1, 0.4)

    # --- Build ---
    model = ModernTransformer(
        inp.n_words, out.n_words, d_model, nhead, num_layers, dim_ff, dropout
    ).to(Config.DEVICE)

    ds = TransDataset(train_pairs, inp, out, augment=True)
    loader = DataLoader(
        ds, batch_size=Config.BATCH_SIZE, shuffle=True, collate_fn=DataEngine.collate_fn
    )

    # Pass pct_start to Trainer
    trainer = Trainer(
        model, lr, wd, len(loader), Config.N_EPOCHS_OPT, pct_start=pct_start
    )

    try:
        hist = trainer.fit(loader, val_pairs, inp, out, Config.N_EPOCHS_OPT, trial)
        return hist["bleu"][-1] + hist["meteor"][-1]
    except RuntimeError as e:
        print(f"Trial failed: {e}")
        return 0.0


def check_validation_samples(model, pairs, inp, out, num_samples=5):
    """
    Explicit Validation Sample Check
    """
    print("\n" + "=" * 60)
    print("[Validation] Check Samples (using Best Saved Model):")
    print("=" * 60)

    model.eval()
    subset = random.sample(pairs, min(len(pairs), num_samples))

    for src_txt, tgt_txt in subset:
        # Prepare input
        tokens = src_txt if inp.name == "cn" else src_txt.split()
        s_idx = [inp.word2index.get(w, Config.PAD_TOKEN) for w in tokens]
        src_tensor = torch.tensor(
            s_idx + [Config.EOS_TOKEN], device=Config.DEVICE
        ).unsqueeze(1)

        # Predict
        out_tokens = Evaluator.beam_search(model, src_tensor, out, beam_width=6)
        out_txt = " ".join(out_tokens)

        print(f"SRC: {src_txt}")
        print(f"TGT: {tgt_txt}")
        print(f"OUT: {out_txt}")
        print("-" * 40)
    print("\n")


def main():
    # Step 1: Data Preparation
    print("\n" + "=" * 50 + "\n[Step 1] Loading and Processing Data\n" + "=" * 50)
    inp, out, pairs = DataEngine.prepare_data()
    random.shuffle(pairs)
    split_idx = int(len(pairs) * 0.8)
    train_pairs, val_pairs = pairs[:split_idx], pairs[split_idx:]
    print(f"Train Size: {len(train_pairs)} | Val Size: {len(val_pairs)}")

    # Step 2: Optuna Optimization
    print("\n" + "=" * 50 + "\n[Step 2] Optuna Hyperparameter Search\n" + "=" * 50)
    study = optuna.create_study(direction="maximize")
    study.optimize(
        lambda t: objective(t, inp, out, train_pairs, val_pairs),
        n_trials=Config.N_TRIALS,
    )

    best = study.best_params
    print(f"\n>>> Best Parameters: {best}")

    # Step 3: Build Best Model
    print(
        "\n"
        + "=" * 50
        + "\n[Step 3] Initializing Final Modern Transformer\n"
        + "=" * 50
    )
    d_model = best["base_dim"] * best["nhead"]
    model = ModernTransformer(
        inp.n_words,
        out.n_words,
        d_model,
        best["nhead"],
        best["num_layers"],
        d_model * best["ff_mult"],
        best["dropout"],
    ).to(Config.DEVICE)

    # Step 4: Full Training with SWA
    print(
        "\n"
        + "=" * 50
        + f"\n[Step 4] Full Training ({Config.N_EPOCHS_FULL} Epochs) with SWA\n"
        + "=" * 50
    )
    ds = TransDataset(train_pairs, inp, out, augment=True)
    loader = DataLoader(
        ds,
        batch_size=Config.BATCH_SIZE,
        shuffle=True,
        num_workers=Config.NUM_WORKERS,
        collate_fn=DataEngine.collate_fn,
    )

    # Use best pct_start if available, else default
    best_pct = best.get("pct_start", 0.3)
    trainer = Trainer(
        model,
        best["lr"],
        best["wd"],
        len(loader),
        Config.N_EPOCHS_FULL,
        pct_start=best_pct,
    )
    history = trainer.fit(loader, val_pairs, inp, out, Config.N_EPOCHS_FULL)

    # --- Validation Samples Check ---
    check_validation_samples(trainer.final_model, val_pairs, inp, out, num_samples=5)

    print("\n[Save] Saving the final SWA model...")
    torch.save(trainer.final_model.state_dict(), "final_model_swa.pth")
    print(">>> Model saved to 'final_model_swa.pth'")

    # Step 5: Visualization
    print("\n" + "=" * 50 + "\n[Step 5] Plotting Metrics\n" + "=" * 50)
    plt.figure(figsize=(12, 5), dpi=300)

    plt.subplot(1, 2, 1)
    plt.plot(history["loss"], label="Loss")
    plt.title("Training Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history["bleu"], label="BLEU")
    plt.plot(history["meteor"], label="METEOR")
    if HAS_BERTSCORE:
        x_bert = [i for i, s in enumerate(history["bertscore"]) if s > 0]
        y_bert = [s for s in history["bertscore"] if s > 0]
        plt.plot(x_bert, y_bert, label="BERTScore", marker="o")
    plt.title("Evaluation Metrics")
    plt.legend()

    plt.tight_layout()
    plt.savefig("modern_transformer_results.png")
    print("Saved plot to modern_transformer_results.png")

    # Step 6: Final Inference
    print("\n" + "=" * 50 + "\n[Step 6] Final Inference (SWA Model)\n" + "=" * 50)

    # Use the SWA averaged model for inference
    inference_model = trainer.final_model

    if os.path.exists("test.txt"):
        with (
            open("test.txt", "r", encoding="utf-8") as f,
            open("test_results.txt", "w", encoding="utf-8") as out_f,
        ):
            for line in f:
                line = line.strip()
                if not line:
                    continue

                # Preprocess input
                norm_line = DataEngine.normalize_string(line)
                tokens = norm_line if inp.name == "cn" else norm_line.split()
                src_idx = [inp.word2index.get(w, Config.PAD_TOKEN) for w in tokens]
                src_tensor = torch.tensor(
                    src_idx + [Config.EOS_TOKEN], device=Config.DEVICE
                ).unsqueeze(1)

                # Predict
                pred_words = Evaluator.beam_search(
                    inference_model, src_tensor, out, beam_width=6
                )

                out_f.write(f"{line}\t{' '.join(pred_words)}\n")
        print("Predictions saved to 'test_results.txt'")
    else:
        print("No test.txt found. Performing sanity check on validation set:")
        if isinstance(inference_model, AveragedModel):
            inference_model.module.eval()
        else:
            inference_model.eval()
        for _ in range(5):
            p = random.choice(val_pairs)
            print(f"Src: {p[0]}")
            print(f"Ref: {p[1]}")

            tokens = p[0] if inp.name == "cn" else p[0].split()
            src_idx = [inp.word2index.get(w, Config.PAD_TOKEN) for w in tokens]
            src_tensor = torch.tensor(
                src_idx + [Config.EOS_TOKEN], device=Config.DEVICE
            ).unsqueeze(1)

            pred = Evaluator.beam_search(inference_model, src_tensor, out, beam_width=6)
            print(f"Out: {' '.join(pred)}\n---")


if __name__ == "__main__":
    main()